# GARCH(1,1) Volatility Forecasting
**Multi-Coin Crypto · Sliding-Window Cross-Validation**

---

## Why GARCH for volatility?

Crypto returns exhibit **volatility clustering** — large moves tend to follow large moves. GARCH(1,1) captures this directly:

$$\sigma^2_t = \omega + \alpha \cdot \varepsilon^2_{t-1} + \beta \cdot \sigma^2_{t-1}$$

- $\omega$ → unconditional variance floor (always positive)
- $\alpha$ → *reaction*: how sharply vol jumps after a large return
- $\beta$ → *persistence*: how long elevated vol lingers
- Constraint: $\alpha + \beta < 1$ (stationarity; if ≈1, vol is near-integrated / high persistence)

## Why sliding window CV?

Volatility regimes are temporary — bear markets, euphoria cycles, depegging events. Sliding window avoids diluting recent structure with distant history. If sliding outperforms expanding significantly, that's a diagnostic: the regime changed and old data is actively misleading.

## Evaluation metrics

| Metric | Formula | Why |
|--------|---------|-----|
| **QLIKE** | $\log\hat{\sigma}^2 + RV / \hat{\sigma}^2$ | Robust loss for volatility forecasting (Patton 2011) |
| **MSE (variance)** | $(\hat{\sigma}^2 - RV)^2$ | Standard magnitude error |
| **MAE (vol)** | $|\hat{\sigma} - \sqrt{RV}|$ | Interpretable in return units |
| **Hit rate** | Correct high/low vol regime prediction | Trading signal usefulness |

> **Realized Variance (RV)** proxy = squared next-day return ($r_{t+1}^2$), the standard daily RV estimator when intraday data is unavailable.

In [44]:
from __future__ import annotations

import warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional
from scipy.optimize import minimize, Bounds

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# ── Plotly dark theme ──────────────────────────────────────────────────────
DARK = dict(
    template='plotly_dark',
    paper_bgcolor='#0d1117',
    plot_bgcolor='#0d1117',
    font=dict(family='monospace', color='#c9d1d9'),
)
PALETTE = px.colors.qualitative.Plotly

print('✓ Imports OK')

✓ Imports OK


---
## 1 · Load Data

In [45]:
DATA_PATH = '../data/cv_pool.csv'   # ← adjust if needed

df_raw = pd.read_csv(DATA_PATH, parse_dates=['Date'])
df_raw = df_raw.sort_values(['ticker', 'Date']).reset_index(drop=True)

TICKERS = sorted(df_raw['ticker'].unique())
print(f'Tickers ({len(TICKERS)}): {TICKERS}')
print(f'Total rows: {len(df_raw):,}')
print(f'Date range: {df_raw["Date"].min().date()} → {df_raw["Date"].max().date()}')
df_raw.groupby('ticker')[['Date']].agg(['min','max','count'])

Tickers (6): ['AVAX-USD', 'BNB-USD', 'BTC-USD', 'ETH-USD', 'SOL-USD', 'XRP-USD']
Total rows: 11,999
Date range: 2019-07-18 → 2025-07-07


Date                 
                min        max count
ticker                              
AVAX-USD 2021-04-06 2025-07-07  1554
BNB-USD  2019-07-18 2025-07-07  2182
BTC-USD  2019-07-18 2025-07-07  2182
ETH-USD  2019-07-18 2025-07-07  2182
SOL-USD  2020-10-25 2025-07-07  1717
XRP-USD  2019-07-18 2025-07-07  2182

---
## 2 · GARCH(1,1) Implementation

Pure-numpy MLE — no external GARCH library needed.

In [46]:
@dataclass
class GARCHParams:
    """Fitted GARCH(1,1) parameters."""
    mu:      float   # mean
    omega:   float   # variance floor
    alpha:   float   # ARCH (reaction)
    beta:    float   # GARCH (persistence)
    converged: bool = True
    log_lik:   float = 0.0

    @property
    def persistence(self) -> float:
        return self.alpha + self.beta

    @property
    def half_life(self) -> float:
        """Days for a vol shock to decay to half its size."""
        p = self.persistence
        if p >= 1 or p <= 0:
            return float('inf')
        return np.log(0.5) / np.log(p)

    @property
    def unconditional_vol(self) -> float:
        """Long-run annualised volatility (×√365 for crypto)."""
        denom = 1 - self.persistence
        if denom <= 0:
            return float('nan')
        return np.sqrt(self.omega / denom) * np.sqrt(365)


def _garch_variance_path(
    returns: np.ndarray,
    mu: float,
    omega: float,
    alpha: float,
    beta: float,
) -> np.ndarray:
    """Recursively compute the conditional variance path σ²_t."""
    n = len(returns)
    eps = returns - mu
    sigma2 = np.empty(n)
    # initialise with sample variance
    sigma2[0] = np.var(eps)
    for t in range(1, n):
        sigma2[t] = omega + alpha * eps[t-1]**2 + beta * sigma2[t-1]
    # clip to avoid numerical explosion
    return np.clip(sigma2, 1e-12, None)


def _neg_log_likelihood(
    params: np.ndarray,
    returns: np.ndarray,
) -> float:
    mu, omega, alpha, beta = params
    if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 1:
        return 1e10
    sigma2 = _garch_variance_path(returns, mu, omega, alpha, beta)
    eps = returns - mu
    nll = 0.5 * np.sum(np.log(sigma2) + eps**2 / sigma2)
    return nll if np.isfinite(nll) else 1e10


def fit_garch(returns: np.ndarray, min_obs: int = 100) -> Optional[GARCHParams]:
    """
    Fit GARCH(1,1) via MLE with multiple restarts.
    Returns None if the series is too short or optimisation fails.
    """
    returns = np.asarray(returns, dtype=float)
    returns = returns[np.isfinite(returns)]
    if len(returns) < min_obs:
        return None

    var0   = np.var(returns)
    mu0    = np.mean(returns)

    # Multiple starting points to escape local minima
    init_grid = [
        [mu0,  var0 * 0.05, 0.10, 0.85],
        [mu0,  var0 * 0.05, 0.05, 0.90],
        [mu0,  var0 * 0.10, 0.15, 0.80],
        [0.0,  var0 * 0.02, 0.08, 0.88],
    ]

    best_result = None
    best_nll    = 1e10

    bounds = Bounds(
        lb=[-0.1, 1e-8, 1e-6, 1e-6],
        ub=[ 0.1, var0 * 10, 0.5, 0.999],
    )

    for x0 in init_grid:
        try:
            res = minimize(
                _neg_log_likelihood,
                x0=x0,
                args=(returns,),
                method='L-BFGS-B',
                bounds=bounds,
                options={'maxiter': 1000, 'ftol': 1e-10},
            )
            if res.fun < best_nll:
                best_nll    = res.fun
                best_result = res
        except Exception:
            continue

    if best_result is None or not best_result.success:
        # Return a fallback with sample variance (ARCH(0) = constant)
        var_fallback = np.var(returns)
        return GARCHParams(
            mu=mu0, omega=var_fallback, alpha=0.0, beta=0.0,
            converged=False, log_lik=-best_nll,
        )

    mu, omega, alpha, beta = best_result.x
    return GARCHParams(
        mu=mu, omega=omega, alpha=alpha, beta=beta,
        converged=True, log_lik=-best_nll,
    )


def forecast_variance(
    returns: np.ndarray,
    params: GARCHParams,
    h: int = 1,
) -> np.ndarray:
    """
    Compute one-step-ahead conditional variance forecasts for the whole series.
    Returns σ²_{t+1} for each t.  Last h values are multi-step forecasts
    using the analytical recursion.

    For h=1: σ²_{t+1|t} = ω + α·ε²_t + β·σ²_t
    For h>1:  σ²_{t+h|t} = ω·(1 + p + p² + … + p^{h-2}) + p^{h-1}·σ²_{t+1|t}
             where p = α + β
    """
    sigma2 = _garch_variance_path(
        returns, params.mu, params.omega, params.alpha, params.beta
    )
    eps = returns - params.mu

    # One-step-ahead: shift forward by 1
    forecasts_1 = np.empty(len(returns))
    forecasts_1[0] = sigma2[0]
    for t in range(1, len(returns)):
        forecasts_1[t] = params.omega + params.alpha * eps[t-1]**2 + params.beta * sigma2[t-1]

    if h == 1:
        return np.clip(forecasts_1, 1e-12, None)

    # Multi-step analytical expansion
    p = params.persistence
    if p < 1:
        # sum of geometric series: ω·(1 - p^{h-1})/(1 - p) + p^{h-1}·σ²_{t+1|t}
        geo_sum = params.omega * (1 - p**(h-1)) / (1 - p)
        forecasts_h = geo_sum + p**(h-1) * forecasts_1
    else:
        forecasts_h = forecasts_1  # unit-root: best guess is current forecast

    return np.clip(forecasts_h, 1e-12, None)


print('✓ GARCH(1,1) implementation ready')

✓ GARCH(1,1) implementation ready


---
## 3 · Sliding-Window Cross-Validation

Each fold: fit GARCH on the training window → evaluate on the val window.  
The val window immediately follows training (no gap by default; you can add an embargo).

In [47]:
@dataclass
class GARCHFoldResult:
    ticker:      str
    fold_idx:    int
    train_start: str
    train_end:   str
    val_start:   str
    val_end:     str
    n_train:     int
    n_val:       int
    params:      GARCHParams
    qlike:       float
    mse_var:     float
    mae_vol:     float
    hit_rate:    float   # high-vol regime prediction accuracy
    # per-period arrays (for plotting)
    val_dates:   np.ndarray = field(repr=False, default_factory=lambda: np.array([]))
    val_rv:      np.ndarray = field(repr=False, default_factory=lambda: np.array([]))
    val_sigma2:  np.ndarray = field(repr=False, default_factory=lambda: np.array([]))


def qlike_loss(sigma2_hat: np.ndarray, rv: np.ndarray) -> float:
    """QLIKE: standard robust loss for volatility forecasts (Patton 2011)."""
    sigma2_hat = np.clip(sigma2_hat, 1e-12, None)
    rv         = np.clip(rv,         1e-12, None)
    return float(np.mean(np.log(sigma2_hat) + rv / sigma2_hat))


def _make_sliding_folds(
    dates: pd.DatetimeIndex,
    n_folds: int = 5,
    train_frac: float = 0.40,
    val_frac: float = 0.10,
    gap_days: int = 0,
) -> list[dict]:
    """Return list of {train_idx, val_idx} dicts."""
    dates = dates.sort_values()
    n = len(dates)
    train_size = max(1, int(n * train_frac))
    val_size   = max(1, int(n * val_frac))
    window     = train_size + gap_days + val_size

    if window > n:
        raise ValueError(
            f'Not enough data: window={window} > n={n}. '
            f'Reduce train_frac/val_frac or n_folds.'
        )

    first_start = 0
    last_start  = n - window

    if last_start < first_start:
        last_start = first_start

    starts = np.linspace(first_start, last_start, n_folds, dtype=int)

    folds = []
    for i, s in enumerate(starts):
        te = s + train_size - 1
        vs = te + 1 + gap_days
        ve = min(vs + val_size - 1, n - 1)
        folds.append(dict(
            fold_idx    = i,
            train_start = dates[s],
            train_end   = dates[te],
            val_start   = dates[vs],
            val_end     = dates[ve],
            train_slice = slice(s, te + 1),
            val_slice   = slice(vs, ve + 1),
        ))
    return folds


def cross_validate_garch(
    ticker_df:  pd.DataFrame,
    ticker:     str,
    return_col: str  = 'log_close_return_1',
    n_folds:    int  = 5,
    train_frac: float = 0.45,
    val_frac:   float = 0.10,
    gap_days:   int  = 0,
    vol_pct:    float = 0.75,   # percentile for high-vol regime label
    verbose:    bool  = True,
) -> list[GARCHFoldResult]:
    """
    Run sliding-window GARCH CV for a single ticker.

    Realized variance proxy: r_{t+1}^2 (squared next-day return).
    """
    df = ticker_df.sort_values('Date').reset_index(drop=True)
    dates   = pd.DatetimeIndex(df['Date'])
    returns = df[return_col].values.astype(float)

    # RV proxy: squared next-day return (shift back so it aligns with today's forecast)
    rv = np.roll(returns**2, -1)  # rv[t] = r_{t+1}^2
    rv[-1] = rv[-2]               # fill last row

    folds   = _make_sliding_folds(dates, n_folds, train_frac, val_frac, gap_days)
    results = []

    for fold in folds:
        fi = fold['fold_idx']
        ts = fold['train_slice']
        vs = fold['val_slice']

        train_ret = returns[ts]
        val_ret   = returns[vs]
        val_rv    = rv[vs]
        val_dates = np.array(dates[vs])

        if len(train_ret) < 60 or len(val_ret) < 5:
            if verbose:
                print(f'  [{ticker}] Fold {fi}: skipped (too short)')
            continue

        params = fit_garch(train_ret)
        if params is None:
            continue

        # 1-step-ahead forecasts on val window (warm-start with training data)
        combined     = np.concatenate([train_ret, val_ret])
        sigma2_all   = _garch_variance_path(
            combined, params.mu, params.omega, params.alpha, params.beta
        )
        # The val-period forecasts: index offset by len(train)
        val_sigma2_hat = sigma2_all[len(train_ret):]
        # Shift by 1: σ²_{t+1|t}
        forecast_sigma2 = np.empty_like(val_sigma2_hat)
        eps_combined    = combined - params.mu
        for t_rel, t_abs in enumerate(range(len(train_ret), len(combined))):
            if t_abs == 0:
                forecast_sigma2[t_rel] = sigma2_all[0]
            else:
                forecast_sigma2[t_rel] = (
                    params.omega
                    + params.alpha * eps_combined[t_abs-1]**2
                    + params.beta  * sigma2_all[t_abs-1]
                )
        forecast_sigma2 = np.clip(forecast_sigma2, 1e-12, None)

        # Metrics
        ql   = qlike_loss(forecast_sigma2, val_rv)
        mse  = float(np.mean((forecast_sigma2 - val_rv)**2))
        mae  = float(np.mean(np.abs(np.sqrt(forecast_sigma2) - np.sqrt(np.clip(val_rv,0,None)))))

        # Hit rate: predict whether tomorrow is a high-vol day
        vol_threshold = np.percentile(returns**2, vol_pct * 100)
        actual_hv  = (val_rv > vol_threshold).astype(int)
        pred_hv    = (forecast_sigma2 > vol_threshold).astype(int)
        hit_rate   = float(np.mean(actual_hv == pred_hv))

        if verbose:
            print(
                f'  [{ticker}] Fold {fi} '
                f'[{fold["train_start"].date()} → {fold["train_end"].date()}] '
                f'val [{fold["val_start"].date()} → {fold["val_end"].date()}] '
                f'| α={params.alpha:.3f} β={params.beta:.3f} '
                f'persist={params.persistence:.3f} '
                f'| QLIKE={ql:.4f} hit={hit_rate:.3f}'
            )

        results.append(GARCHFoldResult(
            ticker      = ticker,
            fold_idx    = fi,
            train_start = str(fold['train_start'].date()),
            train_end   = str(fold['train_end'].date()),
            val_start   = str(fold['val_start'].date()),
            val_end     = str(fold['val_end'].date()),
            n_train     = len(train_ret),
            n_val       = len(val_ret),
            params      = params,
            qlike       = ql,
            mse_var     = mse,
            mae_vol     = mae,
            hit_rate    = hit_rate,
            val_dates   = val_dates,
            val_rv      = val_rv,
            val_sigma2  = forecast_sigma2,
        ))

    return results


print('✓ CV harness ready')

✓ CV harness ready


---
## 4 · Run CV Across All Tickers

Config:
- **5 sliding folds** per ticker
- **45% training window** (~fixed ~900 days for BTC)
- **10% val window** (~200 days)
- **Gap = 1 day** embargo to avoid lookahead on the boundary

In [48]:
CV_CONFIG = dict(
    n_folds    = 5,
    train_frac = 0.45,
    val_frac   = 0.10,
    gap_days   = 1,
    vol_pct    = 0.75,    # top 25% daily moves = 'high vol'
    verbose    = True,
)

all_results: dict[str, list[GARCHFoldResult]] = {}

for ticker in TICKERS:
    print(f'\n{'─'*60}')
    print(f'  {ticker}')
    print(f'{'─'*60}')
    t_df = df_raw[df_raw['ticker'] == ticker].copy()
    try:
        results = cross_validate_garch(t_df, ticker, **CV_CONFIG)
        all_results[ticker] = results
    except Exception as e:
        print(f'  ✗ {ticker} failed: {e}')
        all_results[ticker] = []

print('\n✓ CV complete')


────────────────────────────────────────────────────────────
  AVAX-USD
────────────────────────────────────────────────────────────
  [AVAX-USD] Fold 0 [2021-04-06 → 2023-03-05] val [2023-03-07 → 2023-08-08] | α=0.162 β=0.808 persist=0.969 | QLIKE=-5.6386 hit=0.800
  [AVAX-USD] Fold 1 [2021-09-27 → 2023-08-26] val [2023-08-28 → 2024-01-29] | α=0.202 β=0.773 persist=0.975 | QLIKE=-5.2198 hit=0.716
  [AVAX-USD] Fold 2 [2022-03-21 → 2024-02-17] val [2024-02-19 → 2024-07-22] | α=0.240 β=0.736 persist=0.976 | QLIKE=-4.9218 hit=0.703
  [AVAX-USD] Fold 3 [2022-09-12 → 2024-08-10] val [2024-08-12 → 2025-01-13] | α=0.188 β=0.713 persist=0.901 | QLIKE=-4.9598 hit=0.703
  [AVAX-USD] Fold 4 [2023-03-06 → 2025-02-01] val [2025-02-03 → 2025-07-07] | α=0.097 β=0.878 persist=0.975 | QLIKE=-4.9679 hit=0.703

────────────────────────────────────────────────────────────
  BNB-USD
────────────────────────────────────────────────────────────
  [BNB-USD] Fold 0 [2019-07-18 → 2022-03-24] val [2022-03-26 → 

---
## 5 · Summary Table

In [49]:
rows = []
for ticker, folds in all_results.items():
    if not folds:
        continue
    for r in folds:
        rows.append(dict(
            ticker      = r.ticker,
            fold        = r.fold_idx,
            train_start = r.train_start,
            train_end   = r.train_end,
            val_start   = r.val_start,
            val_end     = r.val_end,
            n_train     = r.n_train,
            n_val       = r.n_val,
            mu          = round(r.params.mu, 5),
            omega       = round(r.params.omega, 7),
            alpha       = round(r.params.alpha, 4),
            beta        = round(r.params.beta, 4),
            persistence = round(r.params.persistence, 4),
            half_life_d = round(r.params.half_life, 1),
            uncond_vol  = round(r.params.unconditional_vol, 3),
            converged   = r.params.converged,
            qlike       = round(r.qlike, 4),
            mse_var     = round(r.mse_var, 8),
            mae_vol     = round(r.mae_vol, 5),
            hit_rate    = round(r.hit_rate, 3),
        ))

cv_df = pd.DataFrame(rows)

print('Per-fold results:')
display(cv_df)

print('\nMean across folds per ticker:')
summary = (
    cv_df
    .groupby('ticker')[['alpha','beta','persistence','half_life_d',
                         'uncond_vol','qlike','mae_vol','hit_rate']]
    .agg(['mean','std'])
    .round(4)
)
display(summary)

Per-fold results:


,ticker,fold,train_start,train_end,val_start,val_end,n_train,n_val,mu,omega,alpha,beta,persistence,half_life_d,uncond_vol,converged,qlike,mse_var,mae_vol,hit_rate
0,AVAX-USD,0,2021-04-06,2023-03-05,2023-03-07,2023-08-08,699,155,-0.00217,0.000207,0.1615,0.8077,0.9692,22.2,1.568,True,-5.6386,0.000009,0.02755,0.800
1,AVAX-USD,1,2021-09-27,2023-08-26,2023-08-28,2024-01-29,699,155,-0.00353,0.000134,0.2016,0.7731,0.9748,27.1,1.392,True,-5.2198,0.000024,0.02943,0.716
2,AVAX-USD,2,2022-03-21,2024-02-17,2024-02-19,2024-07-22,699,155,-0.00245,0.000138,0.2400,0.7360,0.9760,28.5,1.448,True,-4.9218,0.000020,0.02910,0.703
3,AVAX-USD,3,2022-09-12,2024-08-10,2024-08-12,2025-01-13,699,155,-0.00155,0.000234,0.1881,0.7131,0.9012,6.7,0.930,True,-4.9598,0.000023,0.02598,0.703
4,AVAX-USD,4,2023-03-06,2025-02-01,2025-02-03,2025-07-07,699,155,-0.00111,0.000062,0.0969,0.8780,0.9749,27.3,0.948,True,-4.9679,0.000019,0.02794,0.703
5,BNB-USD,0,2019-07-18,2022-03-24,2022-03-26,2022-10-29,981,218,0.00215,0.000109,0.1759,0.8138,0.9897,66.7,1.964,True,-5.7323,0.000011,0.02476,0.500
6,BNB-USD,1,2020-03-19,2022-11-24,2022-11-26,2023-07-01,981,218,0.00184,0.000068,0.1364,0.8499,0.9863,50.2,1.350,True,-6.2325,0.000003,0.02069,0.702
7,BNB-USD,2,2020-11-20,2023-07-28,2023-07-30,2024-03-03,981,218,0.00078,0.000053,0.1555,0.8387,0.9942,119.8,1.837,True,-6.5231,0.000002,0.01816,0.821
8,BNB-USD,3,2021-07-23,2024-03-29,2024-03-31,2024-11-03,981,218,0.00087,0.000051,0.1617,0.8146,0.9764,29.0,0.889,True,-6.2231,0.000002,0.01682,0.752
9,BNB-USD,4,2022-03-26,2024-11-30,2024-12-02,2025-07-07,981,218,0.00064,0.000098,0.2190,0.6972,0.9162,7.9,0.652,True,-6.3255,0.000003,0.01808,0.766



Mean across folds per ticker:


alpha            beta         persistence         half_life_d  \
            mean     std    mean     std        mean     std        mean   
ticker                                                                     
AVAX-USD  0.1776  0.0533  0.7816  0.0648      0.9592  0.0325       22.36   
BNB-USD   0.1697  0.0310  0.8028  0.0611      0.9726  0.0322       54.72   
BTC-USD   0.0961  0.0593  0.8093  0.1296      0.9054  0.0716       12.24   
ETH-USD   0.0923  0.0208  0.8721  0.0363      0.9644  0.0210       28.84   
SOL-USD   0.2481  0.1134  0.4998  0.2702      0.7478  0.1655        5.02   
XRP-USD   0.1946  0.1898  0.4144  0.3190      0.6091  0.3818         inf   

                  uncond_vol            qlike         mae_vol          \
              std       mean      std    mean     std    mean     std   
ticker                                                                  
AVAX-USD   9.0806     1.2572   0.2974 -5.1416  0.3019  0.0280  0.0014   
BNB-USD   42.5822     1.3384   0.5730 -6.2073  0.2916  0.0197  0.0032   
BTC-USD    9.8602     0.6592   0.0895 -6.2803  0.2783  0.0190  0.0030   
ETH-USD   21.6012     0.9016   0.1786 -5.7275  0.5303  0.0235  0.0044   
SOL-USD    6.3629     1.1548   0.2309 -5.1335  0.2036  0.0315  0.0063   
XRP-USD       NaN    15.1426  31.2816 -5.4382  0.3572  0.0323  0.0070   

         hit_rate          
             mean     std  
ticker                     
AVAX-USD   0.7250  0.0423  
BNB-USD    0.7082  0.1239  
BTC-USD    0.5348  0.2246  
ETH-USD    0.6478  0.1498  
SOL-USD    0.6984  0.1585  
XRP-USD    0.3366  0.2061

---
## 6 · Visualisations

### 6.1 GARCH Parameters Across Folds

In [50]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['α (Reaction)', 'β (Persistence)', 'α + β (Total Persistence)'],
    horizontal_spacing=0.08,
)

for i, ticker in enumerate(TICKERS):
    t_df = cv_df[cv_df['ticker'] == ticker]
    color = PALETTE[i % len(PALETTE)]
    kw = dict(name=ticker, legendgroup=ticker,
              marker_color=color, showlegend=True, boxpoints='all',
              pointpos=0, jitter=0.3)
    fig.add_trace(go.Box(y=t_df['alpha'],       x=[ticker]*len(t_df), **kw), row=1, col=1)
    kw['showlegend'] = False
    fig.add_trace(go.Box(y=t_df['beta'],        x=[ticker]*len(t_df), **kw), row=1, col=2)
    fig.add_trace(go.Box(y=t_df['persistence'], x=[ticker]*len(t_df), **kw), row=1, col=3)

# Stationarity line at persistence = 1
fig.add_hline(y=0.99, row=1, col=3, line_dash='dash', line_color='red',
              annotation_text='unit root', annotation_position='top right')

fig.update_layout(
    title='GARCH(1,1) Parameter Distribution Across Sliding-Window Folds',
    height=420,
    **DARK,
)
fig.show()

### 6.2 Validation Metrics Heatmap

In [51]:
metrics = ['qlike', 'mae_vol', 'hit_rate']
metric_labels = ['QLIKE (↓)', 'MAE Vol (↓)', 'Hit Rate (↑)']

mean_df = cv_df.groupby('ticker')[metrics].mean()

# Normalise for display: good direction → green
# QLIKE & MAE: lower is better → invert
# Hit rate: higher is better
norm = mean_df.copy()
for col in ['qlike', 'mae_vol']:
    col_range = norm[col].max() - norm[col].min()
    norm[col] = 1 - (norm[col] - norm[col].min()) / (col_range + 1e-12)
for col in ['hit_rate']:
    col_range = norm[col].max() - norm[col].min()
    norm[col] = (norm[col] - norm[col].min()) / (col_range + 1e-12)

fig = go.Figure(data=go.Heatmap(
    z=norm.values,
    x=metric_labels,
    y=norm.index.tolist(),
    text=mean_df.values.round(4),
    texttemplate='%{text}',
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    showscale=True,
))
fig.update_layout(
    title='Mean Val Metrics per Ticker  (green = better)',
    height=320,
    **DARK,
)
fig.show()

### 6.3 QLIKE & Hit Rate by Fold — Stability Check

In [52]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['QLIKE by Fold (↓ better)', 'Hit Rate by Fold (↑ better)'],
    horizontal_spacing=0.10,
)

for i, ticker in enumerate(TICKERS):
    t_df = cv_df[cv_df['ticker'] == ticker].sort_values('fold')
    color = PALETTE[i % len(PALETTE)]
    kw = dict(x=t_df['fold'], name=ticker, legendgroup=ticker,
              line_color=color, mode='lines+markers',
              marker=dict(size=7, symbol='circle-open'))
    fig.add_trace(go.Scatter(y=t_df['qlike'],    **kw, showlegend=True),  row=1, col=1)
    fig.add_trace(go.Scatter(y=t_df['hit_rate'], **kw, showlegend=False), row=1, col=2)

# Random baseline at 0.5 for hit rate
fig.add_hline(y=0.5, row=1, col=2, line_dash='dot', line_color='grey',
              annotation_text='random', annotation_position='bottom right')

fig.update_xaxes(title_text='Fold', tickmode='array', tickvals=list(range(CV_CONFIG['n_folds'])))
fig.update_layout(height=380, title='CV Metrics Across Folds (Sliding Window)', **DARK)
fig.show()

### 6.4 Vol Forecast vs Realised — Last Fold per Ticker

In [53]:
n_tickers = len(TICKERS)
ncols = 2
nrows = (n_tickers + ncols - 1) // ncols

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=TICKERS,
    shared_xaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

for i, ticker in enumerate(TICKERS):
    row = i // ncols + 1
    col = i  % ncols + 1
    color = PALETTE[i % len(PALETTE)]

    folds = all_results.get(ticker, [])
    if not folds:
        continue

    # Use the last fold (most recent out-of-sample period)
    last = max(folds, key=lambda r: r.fold_idx)

    dates   = pd.to_datetime(last.val_dates)
    rv_vol  = np.sqrt(np.clip(last.val_rv, 0, None))
    hat_vol = np.sqrt(last.val_sigma2)

    fig.add_trace(
        go.Scatter(x=dates, y=rv_vol * 100,
                   name='|r_{t+1}| (realised)',
                   line=dict(color='#8b949e', width=1),
                   legendgroup=ticker, showlegend=(i == 0)),
        row=row, col=col,
    )
    fig.add_trace(
        go.Scatter(x=dates, y=hat_vol * 100,
                   name='σ̂ GARCH (forecast)',
                   line=dict(color=color, width=2),
                   legendgroup=ticker, showlegend=(i == 0)),
        row=row, col=col,
    )

    # Annotate persistence
    p = last.params.persistence
    fig.add_annotation(
        x=0.02, y=0.95, xref='paper' if i==0 else f'x{i+1} domain',
        # yref=f'y{i+1} domain',
        text=f'α+β={p:.3f}  HL={last.params.half_life:.0f}d',
        showarrow=False, font=dict(size=10, color='#e6db74'),
        # axref=f'x{i+1} domain', ayref=f'y{i+1} domain',
    )

fig.update_yaxes(title_text='Daily Vol (%)')
fig.update_layout(
    title='GARCH Forecast vs Realised Vol — Last Sliding-Window Fold',
    height=280 * nrows,
    **DARK,
)
fig.show()

### 6.5 Regime Persistence Map

In [54]:
# α vs β scatter — shows where each ticker sits in the GARCH 'triangle'
# Stationarity region: α + β < 1  (triangle below the diagonal)
mean_params = cv_df.groupby('ticker')[['alpha','beta','persistence','half_life_d']].mean().reset_index()

fig = go.Figure()

# Stationarity boundary
alpha_line = np.linspace(0, 0.5, 100)
fig.add_trace(go.Scatter(
    x=alpha_line, y=1 - alpha_line,
    mode='lines', name='unit root (α+β=1)',
    line=dict(color='red', dash='dash', width=1.5),
))

for i, row in mean_params.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['alpha']], y=[row['beta']],
        mode='markers+text',
        name=row['ticker'],
        text=[row['ticker']],
        textposition='top right',
        marker=dict(size=14, color=PALETTE[i % len(PALETTE)],
                    line=dict(width=1.5, color='white')),
        showlegend=True,
    ))

fig.update_layout(
    title='GARCH Parameter Space  (α = reaction, β = persistence)<br><sup>Closer to the unit-root line → longer-lasting vol shocks</sup>',
    xaxis_title='α (ARCH reaction)',
    yaxis_title='β (GARCH persistence)',
    height=480,
    xaxis=dict(range=[0, 0.55]),
    yaxis=dict(range=[0.4, 1.05]),
    **DARK,
)
fig.show()

### 6.6 Full Conditional Volatility Path (All History)

In [55]:
fig = go.Figure()

for i, ticker in enumerate(TICKERS):
    t_df = df_raw[df_raw['ticker'] == ticker].sort_values('Date').reset_index(drop=True)
    returns = t_df['log_close_return_1'].values.astype(float)
    dates   = pd.to_datetime(t_df['Date'])

    # Fit on all data to get full conditional vol path
    params = fit_garch(returns)
    if params is None:
        continue

    sigma2 = _garch_variance_path(returns, params.mu, params.omega, params.alpha, params.beta)
    ann_vol = np.sqrt(sigma2) * np.sqrt(365) * 100   # annualised %

    fig.add_trace(go.Scatter(
        x=dates, y=ann_vol,
        name=ticker,
        line=dict(color=PALETTE[i % len(PALETTE)], width=1.5),
        mode='lines',
    ))

fig.update_layout(
    title='Conditional Annualised Volatility Path — GARCH(1,1) Fitted on Full History',
    xaxis_title='Date',
    yaxis_title='Annualised Vol (%)',
    height=440,
    **DARK,
)
fig.show()

---
## 7 · Expanding vs Sliding Comparison

For BTC (most data), test whether expanding window outperforms sliding — a diagnostic of how stable the vol regime is.

In [56]:
def cross_validate_garch_strategy(
    ticker_df: pd.DataFrame,
    ticker: str,
    strategy: str,   # 'sliding' or 'expanding'
    **kw,
) -> list[GARCHFoldResult]:
    """Wrapper for expanding-window variant."""
    if strategy == 'sliding':
        return cross_validate_garch(ticker_df, ticker, **kw)

    # Expanding: train_start is always t=0, val_start slides
    df = ticker_df.sort_values('Date').reset_index(drop=True)
    dates   = pd.DatetimeIndex(df['Date'])
    returns = df['log_close_return_1'].values.astype(float)
    rv      = np.roll(returns**2, -1); rv[-1] = rv[-2]

    n          = len(dates)
    n_folds    = kw.get('n_folds', 5)
    val_frac   = kw.get('val_frac', 0.10)
    min_train  = kw.get('train_frac', 0.45)
    gap_days   = kw.get('gap_days', 0)
    vol_pct    = kw.get('vol_pct', 0.75)
    verbose    = kw.get('verbose', False)

    val_size   = max(1, int(n * val_frac))
    min_train_n = max(1, int(n * min_train))
    first_val  = min_train_n + gap_days
    last_val   = n - val_size

    val_starts = np.linspace(first_val, last_val, n_folds, dtype=int)
    results    = []

    for fi, vs in enumerate(val_starts):
        ve       = min(vs + val_size, n) - 1
        te       = vs - 1 - gap_days
        train_ret = returns[:te+1]
        val_ret   = returns[vs:ve+1]
        val_rv_f  = rv[vs:ve+1]
        val_dates = np.array(dates[vs:ve+1])

        if len(train_ret) < 60 or len(val_ret) < 5:
            continue

        params = fit_garch(train_ret)
        if params is None:
            continue

        combined   = np.concatenate([train_ret, val_ret])
        sigma2_all = _garch_variance_path(combined, params.mu, params.omega, params.alpha, params.beta)
        eps_comb   = combined - params.mu
        fc = np.empty(len(val_ret))
        for t_rel, t_abs in enumerate(range(len(train_ret), len(combined))):
            fc[t_rel] = (
                params.omega
                + params.alpha * eps_comb[t_abs-1]**2
                + params.beta  * sigma2_all[t_abs-1]
            ) if t_abs > 0 else sigma2_all[0]
        fc = np.clip(fc, 1e-12, None)

        ql       = qlike_loss(fc, val_rv_f)
        mse      = float(np.mean((fc - val_rv_f)**2))
        mae      = float(np.mean(np.abs(np.sqrt(fc) - np.sqrt(np.clip(val_rv_f,0,None)))))
        vt       = np.percentile(returns**2, vol_pct * 100)
        hit_rate = float(np.mean((fc > vt).astype(int) == (val_rv_f > vt).astype(int)))

        results.append(GARCHFoldResult(
            ticker=ticker, fold_idx=fi,
            train_start=str(dates[0].date()), train_end=str(dates[te].date()),
            val_start=str(dates[vs].date()), val_end=str(dates[ve].date()),
            n_train=len(train_ret), n_val=len(val_ret),
            params=params, qlike=ql, mse_var=mse, mae_vol=mae, hit_rate=hit_rate,
            val_dates=val_dates, val_rv=val_rv_f, val_sigma2=fc,
        ))
    return results


# Run comparison on BTC (longest history)
btc_df = df_raw[df_raw['ticker'] == 'BTC-USD'].copy()
print('Running expanding window for BTC...')
btc_expanding = cross_validate_garch_strategy(btc_df, 'BTC-USD', 'expanding', **{**CV_CONFIG, 'verbose': False})
btc_sliding   = all_results['BTC-USD']

comp = pd.DataFrame([
    dict(strategy='sliding',
         qlike_mean=np.mean([r.qlike for r in btc_sliding]),
         qlike_std =np.std ([r.qlike for r in btc_sliding]),
         hit_mean  =np.mean([r.hit_rate for r in btc_sliding]),
         hit_std   =np.std ([r.hit_rate for r in btc_sliding])),
    dict(strategy='expanding',
         qlike_mean=np.mean([r.qlike for r in btc_expanding]),
         qlike_std =np.std ([r.qlike for r in btc_expanding]),
         hit_mean  =np.mean([r.hit_rate for r in btc_expanding]),
         hit_std   =np.std ([r.hit_rate for r in btc_expanding])),
])

print(comp.to_string(index=False))

# Interpretation
sli_q = comp.loc[comp.strategy=='sliding',   'qlike_mean'].values[0]
exp_q = comp.loc[comp.strategy=='expanding', 'qlike_mean'].values[0]
if sli_q < exp_q - 0.005:
    print('\n→ Sliding wins on QLIKE: vol regime likely shifted; old data confuses the model.')
elif exp_q < sli_q - 0.005:
    print('\n→ Expanding wins on QLIKE: longer history helps; regime is relatively stable.')
else:
    print('\n→ No meaningful difference — both window strategies are equivalent for GARCH.')

Running expanding window for BTC...
 strategy  qlike_mean  qlike_std  hit_mean  hit_std
  sliding   -6.280298   0.248881  0.534862 0.200928
expanding   -6.247954   0.233645  0.452294 0.198101

→ Sliding wins on QLIKE: vol regime likely shifted; old data confuses the model.


---
## 8 · Export GARCH Volatility Signal for DL Models

Export per-ticker GARCH conditional vol as a new feature column.  
This is a **causal** (no-lookahead) signal: σ²_{t+1|t} fitted on data up to day t.

In [57]:
output_dfs = []

for ticker in TICKERS:
    t_df    = df_raw[df_raw['ticker'] == ticker].sort_values('Date').reset_index(drop=True)
    returns = t_df['log_close_return_1'].values.astype(float)

    params = fit_garch(returns)
    if params is None:
        print(f'  ✗ {ticker}: fit failed, filling with rolling std²')
        t_df['garch_sigma2'] = pd.Series(returns).rolling(20).var().values
    else:
        sigma2 = _garch_variance_path(returns, params.mu, params.omega, params.alpha, params.beta)
        # One-step-ahead forecast (shift by 1 so it's available at time t)
        eps    = returns - params.mu
        fc     = np.empty(len(returns))
        fc[0]  = sigma2[0]
        for t in range(1, len(returns)):
            fc[t] = params.omega + params.alpha * eps[t-1]**2 + params.beta * sigma2[t-1]
        t_df['garch_sigma2'] = np.clip(fc, 1e-12, None)
        t_df['garch_vol']    = np.sqrt(t_df['garch_sigma2'])  # daily conditional vol
        t_df['garch_ann_vol'] = t_df['garch_vol'] * np.sqrt(365)  # annualised

        # Regime: 1 if garch_sigma2 > 75th percentile
        q75 = np.percentile(fc, 75)
        t_df['garch_high_vol_regime'] = (t_df['garch_sigma2'] > q75).astype(int)

        print(f'  ✓ {ticker}  α={params.alpha:.3f}  β={params.beta:.3f}  '
              f'persist={params.persistence:.3f}  HL={params.half_life:.0f}d  '
              f'unc_vol={params.unconditional_vol:.1%}')

    output_dfs.append(t_df)

full_output = pd.concat(output_dfs, ignore_index=True)

import os
os.makedirs('./data', exist_ok=True)
full_output.to_csv('./data/cv_pool_with_garch.csv', index=False)
print(f'\n✓ Saved ./data/cv_pool_with_garch.csv  ({len(full_output):,} rows)')
print(f'  New columns: garch_sigma2, garch_vol, garch_ann_vol, garch_high_vol_regime')

# Quick summary
full_output.groupby('ticker')[['garch_ann_vol']].describe().round(3)

  ✓ AVAX-USD  α=0.132  β=0.841  persist=0.973  HL=26d  unc_vol=124.2%
  ✓ BNB-USD  α=0.158  β=0.841  persist=0.999  HL=642d  unc_vol=353.5%
  ✓ BTC-USD  α=0.115  β=0.845  persist=0.960  HL=17d  unc_vol=72.5%
  ✓ ETH-USD  α=0.079  β=0.903  persist=0.982  HL=39d  unc_vol=93.6%
  ✓ SOL-USD  α=0.129  β=0.837  persist=0.966  HL=20d  unc_vol=128.2%
  ✓ XRP-USD  α=0.313  β=0.604  persist=0.916  HL=8d  unc_vol=133.7%

✓ Saved ./data/cv_pool_with_garch.csv  (11,999 rows)
  New columns: garch_sigma2, garch_vol, garch_ann_vol, garch_high_vol_regime


garch_ann_vol                                                 
                 count   mean    std    min    25%    50%    75%    max
ticker                                                                 
AVAX-USD        1554.0  1.063  0.387  0.541  0.815  0.958  1.203  3.461
BNB-USD         2182.0  0.766  0.466  0.330  0.509  0.639  0.853  4.749
BTC-USD         2182.0  0.624  0.207  0.385  0.499  0.580  0.689  3.090
ETH-USD         2182.0  0.801  0.278  0.430  0.627  0.742  0.888  3.141
SOL-USD         1717.0  1.132  0.428  0.655  0.862  1.012  1.244  4.178
XRP-USD         2182.0  0.964  0.513  0.619  0.696  0.794  1.013  6.063

---
## 9 · Key Takeaways

| Insight | What to do |
|---------|------------|
| **α + β near 1** (e.g. > 0.97) | Expect vol shocks to persist for many days; GARCH signal has long memory. Consider using **EGARCH** or **GJR-GARCH** if you see asymmetry (negative returns → bigger vol spikes) |
| **Hit rate > 0.60** | GARCH vol regime flag is predictive — add `garch_high_vol_regime` as a feature to your GRU |
| **Sliding wins QLIKE** | Regime changed — use only recent history for your vol model; consider a shorter training window |
| **Expanding wins QLIKE** | Vol behaviour is stable across the full history; more data helps |
| **Low α, high β** | Vol is slow-moving; regime flags won't flip often; model will trade less frequently |
| **High α, low β** | Vol reacts quickly but mean-reverts fast; suitable for short-horizon signals |

### Integration into your GRU pipeline

The file `./data/cv_pool_with_garch.csv` adds 4 new columns:
- `garch_sigma2` — raw conditional variance (the GARCH output)
- `garch_vol` — daily conditional volatility (√σ²)
- `garch_ann_vol` — annualised conditional vol (garch_vol × √365)
- `garch_high_vol_regime` — binary regime flag (top 25% vol days = 1)

**Recommended usage:**
1. Add `z_garch_vol` (rolling z-scored version) to your `~21 feature` set
2. Add `garch_high_vol_regime` as a binary feature (already 0/1, no scaling needed)
3. Consider using `garch_sigma2` as a **volatility-normalised target**: `target_vnorm = target / garch_vol` — this is equivalent to a Sharpe-like return and reduces the magnitude collapse in recursive prediction